# Spare-it: YOLOv8 Segmentation Model Training 

### Overview
This POC notebook contains the code to convert the Spare-it dataset, augmented datasets, and external datasets to YOLOv8 format, train the YOLOv8 model, and analyze training results. 

### Training Data Setup

The training data setup process involves converting annotations from COCO JSON format to YOLO format, cropping images to only inlcude the categories that we are focusing on, and allocating datasets for training and validation in the format that YOLO expects. More details are provided below.

*NOTE: The file paths used in the code are specific to the SCC file system. In order to reproduce the results, you should set the file paths to the dataset on your system wherever they are used. The code contains comments that should hopefully make this easy. 

#### Process

1. **Annotation Conversion + Cropping**:
   - Convert COCO JSON annotations to YOLO format.
   - Applies category ID mapping using `id_to_index`.
   - Generates YOLO format files in the `labels` directory.
   - Crops images to only include categories specified in `id_to_index` with additional padding

2. **Dataset Allocation**:
   - Creates a YOLO custom dataset structure:
     ```
     ./datasets
     ├── train
     │   ├── images
     │   └── labels
     └── val
         ├── images
         └── labels
     ```
   - Copies image and label files to their respective directories in the dataset structure.
   - Splits the dataset into training and validation. The proportion of data allocated to each set can be specified.
   - Balances the class distribution between training and validation sets to ensure validation metrics are informative. 
   - Ensures external datasets are only added to training set so the validation set is consistent across runs. 


#### Extra Notes

- The `id_to_index` mapping is crucial for matching YOLO class IDs.
- YOLO annotations use 16 decimal places for accuracy, consistent with formats used by platforms like Roboflow.
- The dataset allocation process supports various image extensions: '.jpg', '.jpeg', '.png', '.gif'.
- The script prints the total number of files in train and validation sets for verification.
- If no matching image is found for a label file during dataset allocation, a message is printed to alert the user.

### Annotation Conversion + Cropping
As mentioned above, we will start by converting annotations from cocojson to YOLO format. The code block below is where we specify the directories where we will save the formatted images and labels and delete them if they already exist. 

In [1]:
import os
import shutil

#These are the paths where the formatted data will be saved. By default they will be saved in the working directory. 
labels_dir = './labels'
images_cropped_dir = './images_cropped'

def delete_directory(path):
    if os.path.exists(path):
        shutil.rmtree(path)

#Delete formatted directories if they exist already
delete_directory(labels_dir)
delete_directory(images_cropped_dir)

Next we will convert the cocojson annotations to YOLO format and crop the images to only include the categories defined in `id_to_index`. In order to use this code, you will need to set `coco_dir` and `images_dir` file paths to where the dataset is located on your file system. You can also set the `PADDING_PERCENTAGE` to adjust the background for cropping as needed. 

One of our key contributions this semester is integrating external datasets. Specifically, we provide code for transforming the TACO dataset to the Spare-it format as well as a standard for reconciling the different labeling conventions used. We also provide code for creating an augmented version of the original Spare-it dataset using the copy-paste augmentation. In order to add either of these datasets for training, you can simply run the code block below multiple times and change `coco_dir` and `images_dir` for each dataset you would like to integrate. 

In [2]:
import json
from PIL import Image, ImageOps
from tqdm import tqdm 

# SET THE PATH TO YOUR DATASET HERE
coco_dir = '/projectnb/ds549/datasets/spare-it/original_dataset/cocojson'
images_dir = '/projectnb/ds549/datasets/spare-it/original_dataset/images'

#This is the percentage of padding that will be added when cropping only the categories we would like to focus on. 
PADDING_PERCENTAGE = 0.2

#The accepeted image extensions
image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.JPG']

# Full id_to_index values (matches with dataset.yaml)
# 92 Clean Plastic Film, 106 Plastic Wrap - Class Merge, 131 - No Foodshare
# id_to_index = {1: 0, 2: 1, 4: 2, 5: 3, 6: 4, 7: 5, 8: 6, 9: 7, 10: 8, 11: 9, 14: 10, 15: 11, 16: 12, 
#                17: 13, 18: 14, 19: 15, 20: 16, 21: 17, 22: 18, 23: 19, 24: 20, 25: 21, 26: 22, 27: 23, 
#                30: 24, 36: 25, 38: 26, 40: 27, 43: 28, 44: 29, 45: 30, 46: 31, 47: 32, 48: 33, 49: 34, 
#                50: 35, 51: 36, 52: 37, 53: 38, 54: 39, 55: 40, 56: 41, 57: 42, 58: 43, 59: 44, 60: 45, 
#                63: 46, 65: 47, 66: 48, 67: 49, 68: 50, 69: 51, 70: 52, 72: 53, 73: 54, 76: 55, 77: 56, 
#                78: 57, 79: 58, 80: 59, 82: 60, 83: 61, 84: 62, 85: 63, 86: 64, 87: 65, 88: 66, 89: 67, 
#                91: 68, 92: 69, 106: 69, 93: 70, 94: 71, 95: 72, 96: 73, 97: 74, 98: 75, 99: 76, 100: 77, 101: 78, 
#                102: 79, 103: 80, 107: 81, 109: 82, 110: 83, 112: 84, 114: 85, 115: 86, 116: 87, 
#                117: 88, 118: 89, 119: 90, 120: 91, 121: 92, 122: 93, 123: 94, 124: 95, 125: 96, 126: 97, 
#                127: 98, 128: 99, 129: 100, 130: 101}

# Selected id_to_index values (matches with dataset-parsed.yaml)
# 92 Clean Plastic Film, 106 Plastic Wrap - Class Merge
id_to_index = {
    1: 0, 2: 1, 6: 2, 16: 3, 20: 4, 21: 5, 24: 6,
    25: 7, 27: 8, 38: 9, 51: 10, 52: 11, 55: 12,
    56: 13, 58: 14, 63: 15, 65: 16, 69: 17, 72: 18, 73: 19,
    76: 20, 77: 21, 78: 22, 79: 23, 80: 24, 82: 25, 83: 26, 84: 27, 85: 28, 86: 29,
    89: 30, 91: 31, 92: 32, 93: 33, 96: 34, 98: 35,
    99: 36, 101: 37, 116: 38, 118: 39,
    123: 40, 126: 41, 128: 42
}


def find_crop_bbox(data, id_to_index):
    min_x, min_y = float('inf'), float('inf')
    max_x, max_y = float('-inf'), float('-inf')
    
    for annotation in data['annotations']:
        if annotation['category_id'] in id_to_index:
            if 'bbox' in annotation:
                bbox = annotation['bbox']
                x, y, w, h = bbox
                min_x = min(min_x, x)
                min_y = min(min_y, y)
                max_x = max(max_x, x + w)
                max_y = max(max_y, y + h)
            elif 'segmentation' in annotation:
                segmentation = annotation['segmentation']
                # Extracting min and max x and y from the segmentation points if bbox is not available
                x_coords = segmentation[0][0::2]  # All x-coordinates
                y_coords = segmentation[0][1::2]  # All y-coordinates
                min_x = min(min_x, *x_coords)  
                min_y = min(min_y, *y_coords)
                max_x = max(max_x, *x_coords)
                max_y = max(max_y, *y_coords)
            else:
                break
    
    if min_x == float('inf'):
        return None  # No relevant annotations found
    
    
    padding_x = max((max_x - min_x) * PADDING_PERCENTAGE, 1)  # Ensure at least 1 pixel padding
    padding_y = max((max_y - min_y) * PADDING_PERCENTAGE, 1)  # Ensure at least 1 pixel padding

    image_width = data['images'][0]['width']
    image_height = data['images'][0]['height']

    min_x = max(0, min_x - padding_x)
    min_y = max(0, min_y - padding_y)
    max_x = min(image_width, max_x + padding_x)
    max_y = min(image_height, max_y + padding_y)
    
    return (int(min_x), int(min_y), int(max_x), int(max_y))

def coco_to_yolo(coco_json_path, labels_dir, id_to_index):
    with open(coco_json_path) as f:
        data = json.load(f)
    
    crop_bbox = find_crop_bbox(data, id_to_index)
    if crop_bbox is None:
        return None, False
    
    x_offset, y_offset, max_x, max_y = crop_bbox
    crop_width = max_x - x_offset
    crop_height = max_y - y_offset
    
    if crop_width == 0 or crop_height == 0:
        print(f"Warning: Zero crop dimension in {coco_json_path}. Skipping this file.")
        return None, False
    
    yolo_annotations = []
    
    for annotation in data['annotations']:
        category_id = id_to_index.get(annotation['category_id'], None)
        if category_id is not None:
            segmentation = annotation.get('segmentation', [])
            if segmentation:
                for seg in segmentation:
                    yolo_seg = [f"{((x - x_offset) / crop_width):.16f} {((y - y_offset) / crop_height):.16f}" for x, y in zip(seg[::2], seg[1::2])]
                    yolo_format_seg = f"{category_id} {' '.join(yolo_seg)}"
                    yolo_annotations.append(yolo_format_seg)
    
    if yolo_annotations:
        base_filename = os.path.splitext(os.path.basename(coco_json_path))[0]
        output_filename = os.path.join(labels_dir, f"{base_filename}.txt")
        
        with open(output_filename, 'w') as f:
            for item in yolo_annotations:
                f.write("%s\n" % item)
    
        return crop_bbox, True
    return None, False

def convert_directory(coco_dir, labels_dir, id_to_index):
    if not os.path.exists(labels_dir):
        os.makedirs(labels_dir)
    crop_info = {}
    for filename in tqdm(os.listdir(coco_dir)):
        if filename.endswith(".json"):
            coco_json_path = os.path.join(coco_dir, filename)
            crop_bbox, has_annotations = coco_to_yolo(coco_json_path, labels_dir, id_to_index)
            if has_annotations:
                crop_info[filename] = crop_bbox
    return crop_info

def crop_image(image_path, crop_bbox):
    with Image.open(image_path) as img:
        img = ImageOps.exif_transpose(img)
        cropped_img = img.crop(crop_bbox)
        return cropped_img

# Convert COCO JSON files to YOLO format and get crop bounding boxes
crop_info = convert_directory(coco_dir, labels_dir, id_to_index)

# Crop the images based on bounding boxes
if not os.path.exists(images_cropped_dir):
    os.makedirs(images_cropped_dir)
for json_filename, crop_bbox in tqdm(crop_info.items()):
    for ext in image_extensions:
        image_filename = json_filename.replace('.json', ext)
        image_path = os.path.join(images_dir, image_filename)
        if os.path.exists(image_path):
            cropped_img = crop_image(image_path, crop_bbox)
            cropped_img.save(os.path.join(images_cropped_dir, image_filename))
        
        

print("Conversion and cropping completed.")

100%|██████████| 17868/17868 [18:37<00:00, 16.00it/s]

Conversion and cropping completed.


To verify that we have created the dataset correctly, we will print out the number of images and labels in our original and transformed dataset. Note that we do not expect that these numbers will be exactly the same since we are only focusing on a subset of the categories in Spare-it dataset so the transformed dataset will not contain all of the original data, and if you are integrating multiple external datasets you should run the code block below each time you add a dataset to see that the dataset was correctly added to the transformed dataset. 

In [3]:
# Counts overall files inside cocojson and images and images_cropped and labels to check if conversions were correct
import os

# Function to count files in a directory
def count_files(directory):
    return len([name for name in os.listdir(directory) if os.path.isfile(os.path.join(directory, name))])

cocojson_file_count = count_files(coco_dir)
images_file_count = count_files(images_dir)
images_cropped_file_count = count_files(images_cropped_dir)
labels_file_count = count_files(labels_dir)

# Print the counts
print(f"Number of files in 'cocojson' directory: {cocojson_file_count}")
print(f"Number of files in 'images' directory: {images_file_count}")
print(f"Number of files in 'images_cropped' directory: {images_cropped_file_count}")
print(f"Number of files in 'labels' directory: {labels_file_count}")

Number of files in 'cocojson' directory: 18169
Number of files in 'images' directory: 18169
Number of files in 'images_cropped' directory: 17868
Number of files in 'labels' directory: 17868


### Allocate datasets

Now we will allocate training and validation sets for model training and organize the directories according to the format that YOLO expects (source: https://docs.ultralytics.com/yolov5/tutorials/train_custom_data/?h=custom+dataset). We will try to ensure as much as possible that the class distribution between the training and validation sets are the same as this will likely lead to better training results and more accurate performance metrics. The logic that we use to accomplish this means that the `SPLIT_SIZE` parameter is not the percentage of the data which is allocated for the validation set although the code seems to imply this, but in general increasing `SPLIT_SIZE` will increase the size of the validation set. We find that the best parameters for this process are `SPLIT_SIZE = 0.7`  and `RANDOM_STATE = 326` as this results in close to an 80-20 train test split with all the classes appearing in the validation set. We will also only add images from the original Spare-it dataset to the validation set so that the validation data is consistent even when we integrate external datasets (TACO and/or copy-paste). In order to do this, the logic in the code block below uses the naming conventions for the TACO dataset and the copy-paste dataset which are followed in the code we provide for generating the external datasets. 

In [31]:
import re
import os
import shutil
from collections import defaultdict
from sklearn.model_selection import train_test_split

#These paths should be the same as the paths used earlier for saving the transformed dataset 
labels_path = './labels'
images_path = './images_cropped'

#The path where the final dataset structure will be saved (Default is working directory )
dataset_path = './datasets'

#SPLIT_SIZE (THIS IS NOT THE SAME AS VALIDATION SIZE)
SPLIT_SIZE = 0.7
RANDOM_STATE = 326

image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.JPG']

delete_directory(dataset_path)

subfolders = ['train', 'val']
for subfolder in subfolders:
    os.makedirs(os.path.join(dataset_path, subfolder, 'images'), exist_ok=True)
    os.makedirs(os.path.join(dataset_path, subfolder, 'labels'), exist_ok=True)

def read_labels(file_path):
    with open(file_path, 'r') as f:
        return [line.split()[0] for line in f]

label_files = [file for file in os.listdir(labels_path) if file.endswith('.txt')]

file_classes = defaultdict(list)
class_files = defaultdict(list)
class_instances = defaultdict(int)

for file in label_files:
    file_path = os.path.join(labels_path, file)
    labels = read_labels(file_path)
    file_classes[file] = labels
    for label in labels:
        class_files[label].append(file)
        class_instances[label] += 1

train_files = set()
val_files = set()

for class_label, files in class_files.items():
    if len(files) < 2:
        train_files.update(files)
    else:
        #some files might be from augmented datsets, these ones should go to train set only to ensure the validation data is consistent
        filtered_files = []
        for file in files:
            #taco dataset naming convention uses the word 'batch'
            if 'batch' in file:
                train_files.add(file)
            #our copy-paste dataset uses only numbers for naming
            elif re.fullmatch(r"\d+\.txt", file):
                train_files.add(file)
            #it is a file from the original dataset, so we can add in either train or val 
            else:
                filtered_files.append(file)
        class_train, class_val = train_test_split(filtered_files, test_size=SPLIT_SIZE, random_state=RANDOM_STATE)
        train_files.update(class_train)
        val_files.update(class_val)

val_files = val_files - train_files # This makes sure val will not have duplicates from train

def copy_files(files, source_path_images, source_path_labels, target_path_images, target_path_labels):
    for file in files:
        base_filename = os.path.splitext(file)[0]
        label_file_path = os.path.join(source_path_labels, file)

        copied = False
        for ext in image_extensions:
            image_file = base_filename + ext
            if os.path.exists(os.path.join(source_path_images, image_file)):
                shutil.copy(os.path.join(source_path_images, image_file),
                            os.path.join(target_path_images, image_file))
                copied = True
                break  # stops after first match
        if not copied:
            print(f"No matching image found for {file}")
        else:
            shutil.copy(label_file_path, os.path.join(target_path_labels, file))

copy_files(train_files, images_path, labels_path, os.path.join(dataset_path, 'train', 'images'), os.path.join(dataset_path, 'train', 'labels'))
copy_files(val_files, images_path, labels_path, os.path.join(dataset_path, 'val', 'images'), os.path.join(dataset_path, 'val', 'labels'))

print(f"Total train files: {len(train_files)}")
print(f"Total validation files: {len(val_files)}")

def print_class_distribution(files):
    class_counts = defaultdict(int)
    for file in files:
        for label in file_classes[file]:
            class_counts[label] += 1
    
    print("Class distribution (instance count):")
    for label, count in sorted(class_counts.items(), key=lambda x: int(x[0])):
        print(f"Class {label}: {count}")

print("\nTrain set:")
print_class_distribution(train_files)
print("\nValidation set:")
print_class_distribution(val_files)

print("\nTotal instance count:")
for label, count in sorted(class_instances.items(), key=lambda x: int(x[0])):
    print(f"Class {label}: {count}")

Total train files: 14878
Total validation files: 2990

Train set:
Class distribution (instance count):
Class 0: 4264
Class 1: 10495
Class 2: 315
Class 3: 1406
Class 4: 2641
Class 5: 521
Class 6: 44002
Class 7: 1587
Class 8: 472
Class 9: 3177
Class 10: 6659
Class 11: 4486
Class 12: 806
Class 13: 414
Class 14: 515
Class 15: 10233
Class 16: 974
Class 17: 553
Class 18: 149
Class 19: 781
Class 20: 14412
Class 21: 231
Class 22: 5939
Class 23: 267
Class 24: 1147
Class 25: 1046
Class 26: 4329
Class 27: 933
Class 28: 578
Class 29: 702
Class 30: 3397
Class 31: 382
Class 32: 7216
Class 33: 966
Class 34: 2998
Class 35: 1066
Class 36: 4747
Class 37: 2574
Class 38: 2080
Class 39: 435
Class 40: 1209
Class 41: 2220
Class 42: 1194

Validation set:
Class distribution (instance count):
Class 0: 259
Class 1: 758
Class 2: 29
Class 3: 135
Class 4: 196
Class 5: 37
Class 6: 1870
Class 7: 65
Class 8: 39
Class 9: 1
Class 10: 219
Class 11: 196
Class 12: 33
Class 13: 90
Class 14: 117
Class 15: 634
Class 16: 51
Cl

### Segmentation Model Training

Now we are ready to train our segmentation model. Here are the augmentation parameters that can be set for YOLO (source: https://docs.ultralytics.com/usage/cfg/#train-settings)

1. **HSV Hue (`hsv_h=0.0`)**:
   - Controls hue augmentation. A value of `0.0` means no hue changes will be applied to the images during training.

2. **HSV Saturation (`hsv_s=0.0`)**:
   - Controls saturation augmentation. A value of `0.0` means no changes to saturation will be made during training.

3. **HSV Value (`hsv_v=0.0`)**:
   - Controls value (brightness) augmentation. A value of `0.0` indicates no adjustments to brightness.

4. **Degrees (`degrees=0.0`)**:
   - Controls the random rotation of images. Setting this to `0.0` means images will not be rotated.

5. **Translate (`translate=0.0`)**:
   - Controls translation (shifting) of images along the x and y axes. A value of `0.0` means no translation will be applied.

6. **Scale (`scale=0.0`)**:
   - Controls random scaling of images. With `0.0`, no scaling changes will be made.

7. **Shear (`shear=0.0`)**:
   - Controls shear transformation (slanting of images). Setting this to `0.0` disables shear augmentation.

8. **Perspective (`perspective=0.0`)**:
   - Controls perspective warping of images. A value of `0.0` indicates no perspective distortion will be applied.

9. **Flip Up-Down (`flipud=0.0`)**:
   - Controls vertical flipping of images. With `0.0`, images will not be flipped vertically.

10. **Flip Left-Right (`fliplr=0.0`)**:
    - Controls horizontal flipping of images. A value of `0.0` means no horizontal flipping will occur.

11. **BGR (`bgr=0.0`)**:
    - If enabled, would change the color space from RGB to BGR. Setting this to `0.0` keeps the color space in RGB.

12. **Mosaic (`mosaic=0.0`)**:
    - Controls mosaic augmentation, which combines four images into one. A value of `0.0` means mosaic augmentation is disabled.

13. **MixUp (`mixup=0.0`)**:
    - Controls MixUp augmentation, which overlays one image on another. Setting this to `0.0` disables MixUp.

14. **Copy-Paste (`copy_paste=0.0`)**:
    - Controls Copy-Paste augmentation, which copies objects from one image and pastes them onto another. A value of `0.0` disables this augmentation.

15. **Erasing (`erasing=0.0`)**:
    - Controls random erasing of image patches. A value of `0.0` disables random erasing.

16. **Crop Fraction (`crop_fraction=0.0`)**:
    - Controls cropping of the images. Setting this to `0.0` means images will not be cropped during training.

To use the code block below for training, you should have also created a `dataset.yaml` file according to:

https://docs.ultralytics.com/yolov5/tutorials/train_custom_data/?h=custom+dataset#21-create-datasetyaml

 You will also need to: 
 1. Set the `data` argument to point to your `dataset.yaml` file.
 2. Set the `device` and `workers` arguments appropriately for your computational resources. 
 3. Choose the appropriate YOLO model to use. We use `'yolov8s-seg.pt` by default. 

We will set `epochs=500` and use `patience=5` to stop model training when it is appropriate.

In [6]:
import os
import torch
torch.cuda.empty_cache()

from ultralytics import YOLO
model = YOLO('yolov8s-seg.pt')

results = model.train(
    project="training_results_copypaste_ours", # Destination of runs. If not provided, it will be in runs folder.
    data='/projectnb/ds549/datasets/spare-it/dataset-parsed.yaml', # .yaml file matching your datasets folder data structure
    device= os.getenv("CUDA_VISIBLE_DEVICES"), # default = None
    epochs=500, # default 100
    imgsz=640, # default 640
    batch=16, # Batch size (-1 is auto)
    patience=5, # default 100, Early stopping parameter
    verbose = True, # to see model's accuracy and loss metrics details after each epoch
    workers=10, #number of workers for data loading, default is 8

    # Use this if using multicore
    # device=[0, 1, 2, 3], # ex) 4 core gpu
    # workers=8, #number of workers for data loading, default is 8

    # Filters out low-confidence predictions (controls the quality of object detections).
    # conf=0.35, # None/0.25
    # Controls how Non-Maximum Suppression (NMS) removes overlapping bounding boxes.
    # iou=0.6, # 0.7

    # make sure to have this turned off(uncomment it) if you want custom data augmentation
    # hsv_h=0.0,
    # hsv_s=0.0,
    # hsv_v=0.0,
    # degrees=0.0,
    # translate=0.0,
    # scale=0.0,
    # shear=0.0,
    # perspective=0.0,
    # flipud=0.0,
    # fliplr=0.0,
    # bgr=0.0,
    # mosaic=0.0,
    # mixup=0.0,
    #copy_paste=0.5,
    #copy_paste_mode='mixup'
    # erasing=0.0,
    # crop_fraction=0.0, # end of data augmentations from YOLOv8
)

Ultralytics 8.3.27 🚀 Python-3.10.9 torch-2.5.1+cu124 CUDA:3 (NVIDIA L40S, 45488MiB)
engine/trainer: task=segment, mode=train, model=yolov8s-seg.pt, data=/projectnb/ds549/datasets/spare-it/dataset-parsed.yaml, epochs=500, time=None, patience=5, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=3, workers=10, project=training_results_copypaste_ours, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, sa

train: Scanning /projectnb/ds549/students/ac25/datasets/train/labels... 22336 images, 0 backgrounds, 0 corrupt: 100%|██████████| 22336/22336 [00:23<00:00, 957.90it/s] 

train: WARNING ⚠️ /projectnb/ds549/students/ac25/datasets/train/images/Trash_06e4cb92-5753-4a8a-bdc4-43ffad700922_07bd1196-c632-4bf4-a01f-35660b57d624_465ad689-8383-4ae0-b01f-015197a68b4b.jpeg: 1 duplicate labels removed
train: WARNING ⚠️ /projectnb/ds549/students/ac25/datasets/train/images/Trash_9a8023cc-2608-41e3-82ed-24e7507e91d5_b7e1275d-5d0e-48b8-af8f-254c49833370_c0f2c1b2-aea6-4cab-ba9a-52b06a92963f.jpeg: 1 duplicate labels removed
train: WARNING ⚠️ /projectnb/ds549/students/ac25/datasets/train/images/Trash_c51eb8d5-a003-4877-af91-1c4405cb72ce_8d793a86-560c-44da-8791-733d950fea13_51c72442-b1e9-490f-8a51-e0670186db65.jpeg: 1 duplicate labels removed


train: New cache created: /projectnb/ds549/students/ac25/datasets/train/labels.cache


val: Scanning /projectnb/ds549/students/ac25/datasets/val/labels... 313 images, 0 backgrounds, 0 corrupt: 100%|██████████| 313/313 [00:00<00:00, 2112.61it/s]

...

     66/500      7.02G     0.9143      1.785      1.312      1.132        288        640: 100%|██████████| 1396/1396 [02:01<00:00, 11.49it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 10/10 [00:00<00:00, 10.04it/s]

                   all        313        383      0.675      0.598      0.688      0.603      0.675      0.598      0.676      0.559
EarlyStopping: Training stopped early as no improvement observed in last 5 epochs. Best results observed at epoch 61, best model saved as best.pt.
To update EarlyStopping(patience=5) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



66 epochs completed in 2.261 hours.
Optimizer stripped from training_results_copypaste_ours/train/weights/last.pt, 23.9MB
Optimizer stripped from training_results_copypaste_ours/train/weights/best.pt, 23.9MB

Validating training_results_copypaste_ours/train/weights/best.pt...
Ultralytics 8.3.27 🚀 Python-3.10.9 torch-2.5.1+cu124 CUDA:3 (NVIDIA L40S, 45488MiB)
YOLOv8s-seg summary (fused): 195 layers, 11,796,241 parameters, 0 gradients, 42.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 10/10 [00:01<00:00,  7.02it/s]


                   all        313        383      0.686      0.606      0.693      0.611      0.683      0.604      0.681      0.564
             Paper Cup          2          2      0.759        0.5      0.606      0.545      0.759        0.5      0.606      0.551
Snack or Candy Bag or Wrapper         18         18      0.618      0.539      0.721      0.612      0.618      0.539      0.721      0.618
          Latex Gloves          1          1      0.748          1      0.995      0.995      0.748          1      0.995      0.895
   Shelf Stable Carton          1          1      0.297      0.595      0.497      0.497      0.297      0.595      0.497      0.497
Compostable Fiber Ware          5          5      0.618          1      0.787      0.787      0.618          1      0.787      0.733
   Compostable Cutlery          1          1      0.851          1      0.995      0.895      0.851          1      0.995      0.697
Paper Towel/Napkins/Tissue/Tissue Paper         40         42 


### Training results
Here we evaluate the performance of our trained YOLO model using the following metrics: 

#### Metrics

1. **Precision**:
   - Measures the accuracy of positive predictions.
   - Ratio of true positives to all positive predictions (true positives + false positives).
   - Indicates how well the model avoids labeling negative instances as positive.

2. **Recall**:
   - Measures the completeness of positive predictions.
   - Ratio of true positives to all actual positive instances (true positives + false negatives).
   - Indicates how well the model finds all positive instances.

3. **mAP50**:
   - mAP calculated at an Intersection over Union (IoU) threshold of 0.5.
   - Indicates how well the model predicts and annotates objects with at least 50% overlap.

4. **mAP50-95**:
   - Average of mAP calculated at different IoU thresholds from 0.5 to 0.95 in steps of 0.05.
   - Provides a more robust evaluation across various levels of prediction accuracy.

The Ultralytics package saves detailed metrics and plots for each training run, and we can see the results by running the code block below.

In [2]:
from ultralytics import YOLO
model = YOLO('/projectnb/ds549/students/ac25/training_results_copypaste_ours/train/weights/best.pt') # Set your model path here
results = model.val()

Ultralytics 8.3.27 🚀 Python-3.10.9 torch-2.5.1+cu124 CPU (Intel Xeon E5-2680 v4 2.40GHz)
YOLOv8s-seg summary (fused): 195 layers, 11,796,241 parameters, 0 gradients, 42.5 GFLOPs


val: Scanning /projectnb/ds549/students/ac25/datasets/val/labels.cache... 313 images, 0 backgrounds, 0 corrupt: 100%|██████████| 313/313 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:43<00:00,  2.16s/it]


                   all        313        383      0.691      0.606      0.699      0.613      0.691      0.606      0.685      0.566
             Paper Cup          2          2       0.65        0.5      0.595      0.535       0.65        0.5      0.595      0.541
Snack or Candy Bag or Wrapper         18         18      0.665      0.551      0.717      0.611      0.665      0.551      0.717       0.62
          Latex Gloves          1          1      0.748          1      0.995      0.995      0.748          1      0.995      0.895
   Shelf Stable Carton          1          1      0.296      0.592      0.497      0.497      0.296      0.592      0.497      0.497
Compostable Fiber Ware          5          5      0.513      0.848      0.826      0.826      0.513      0.848      0.826      0.807
   Compostable Cutlery          1          1      0.848          1      0.995      0.895      0.848          1      0.995      0.697
Paper Towel/Napkins/Tissue/Tissue Paper         40         42 